<img src=https://audiovisuales.icesi.edu.co/assets/custom/images/ICESI_logo_prin_descriptor_RGB_POSITIVO_0924.jpg width=200>

# Maestría en Inteligencia Artificial  
## Proyecto de Innovación II  
### Detección de pistolas y cuchillos en actos delictivos mediante visión computacional

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

Este notebook utiliza [Ultralytics](https://docs.ultralytics.com/) para entrenar modelos de detección de objetos YOLO11 con un conjunto de datos personalizado. Al finalizar este Colab, tendremos un modelo YOLO entrenado específicamente para la detección de armas.

**Configuración de entorno (Google Colab o Local)**

In [1]:
# ========================================
# 🔧 FEATURE FLAGS
# ========================================
POC_MODE = True  # Set to False para entrenamiento completo

# Detectar si estamos en Google Colab o entorno local
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Montar Google Drive solo si estamos en Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive'
    print(f"Ejecutando en Google Colab. Ruta base: {BASE_PATH}")
else:
    import os
    BASE_PATH = os.getcwd()  # o puedes especificar una ruta específica
    print(f"Ejecutando en entorno local. Ruta base: {BASE_PATH}")

print(f"POC_MODE: {'ACTIVADO ⚡' if POC_MODE else 'DESACTIVADO (entrenamiento completo)'}")

Ejecutando en entorno local. Ruta base: /Users/gome33773/Documents/MIAA/Weapon-Detection
POC_MODE: ACTIVADO ⚡


**Importación de librerías requeridas**

In [2]:
# Instalación de Ultralytics (YOLO11)
!pip install ultralytics

# === Análisis y manipulación de datos ===
import numpy as np              # Operaciones numéricas con arrays
import pandas as pd             # Manejo de dataframes para análisis de anotaciones

# === Visualización ===
import matplotlib.pyplot as plt # Creación de gráficos y visualizaciones
import seaborn as sns           # Visualizaciones estadísticas mejoradas

# === Procesamiento de imágenes ===
import cv2                      # Lectura de imágenes y dibujo de bounding boxes
from PIL import Image           # Carga de imágenes y obtención de dimensiones

# === Utilidades ===
from tqdm.auto import tqdm      # Barras de progreso para bucles largos
from pathlib import Path        # Manejo moderno de rutas de archivos
from glob import glob           # Búsqueda de archivos con patrones
import xml.etree.ElementTree as ET  # Parseo de anotaciones en formato PASCAL VOC
import zipfile                  # Descompresión del dataset
import os                       # Operaciones del sistema de archivos
import shutil                   # Copiar/mover archivos
import random                   # Shuffle para división train/val

# Configuración de matplotlib para modo interactivo
plt.ion()

print("Setup completo.")

Setup completo.


/Users/gome33773/Documents/MIAA/Proyecto III - Weapon Detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Descomprime las carpetas**

In [3]:
# Definir rutas según el entorno (Colab o local)
if IN_COLAB:
    zip_path = f"{BASE_PATH}/OD-WeaponDetection.zip"
    extract_to = "/content/datasets"
else:
    # Para entorno local, ajusta estas rutas según tu configuración
    zip_path = f"{BASE_PATH}/OD-WeaponDetection.zip"
    extract_to = f"{BASE_PATH}/datasets"

# Descomprimir solo si no existe o está vacío
if not os.path.exists(extract_to) or len(os.listdir(extract_to)) == 0:
    print("Descomprimiendo dataset (primera vez)...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        files = zip_ref.infolist()
        for f in tqdm(files, desc="Progreso", unit="archivo"):
            zip_ref.extract(f, extract_to)

    print("Completado")
else:
    print("Dataset ya existe. No se vuelve a descomprimir.")

Dataset ya existe. No se vuelve a descomprimir.


In [4]:
# Definir carpetas de origen según el entorno
if IN_COLAB:
    SOURCE_DIRS = [
        "/content/datasets/Knife_detection",
        "/content/datasets/Pistol_detection",
        "/content/datasets/Weapons and similar handled objects/Sohas_weapon-Detection"
    ]
    # Carpetas de destino
    FINAL_IMG_DIR = Path("/content/weapon_dataset/images")
    FINAL_ANN_DIR = Path("/content/weapon_dataset/annotations")
else:
    # Para entorno local, ajusta según tu estructura de directorios
    SOURCE_DIRS = [
        f"{extract_to}/Knife_detection",
        f"{extract_to}/Pistol_detection",
        f"{extract_to}/Weapons and similar handled objects/Sohas_weapon-Detection"
    ]
    FINAL_IMG_DIR = Path(f"{BASE_PATH}/weapon_dataset/images")
    FINAL_ANN_DIR = Path(f"{BASE_PATH}/weapon_dataset/annotations")

FINAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ANN_DIR.mkdir(parents=True, exist_ok=True)

# Extensiones de imágenes permitidas
base_ext = ["jpg", "jpeg", "png", "bmp", "tif", "tiff"]

# Construye automáticamente mayúsculas y minúsculas
IMG_EXT = set()
for ext in base_ext:
    IMG_EXT.add(f".{ext}")
    IMG_EXT.add(f".{ext.upper()}")

# Control para evitar duplicados
seen_files = set()

# ================================
#   RECORRER CARPETAS ORIGEN
# ================================
for base in SOURCE_DIRS:
    base_path = Path(base)

    for root, dirs, files in os.walk(base_path):
        for file in files:

            ext = Path(file).suffix
            name_no_ext = Path(file).stem

            # ¿Es imagen?
            if ext in IMG_EXT:
                if name_no_ext not in seen_files:
                    seen_files.add(name_no_ext)
                    shutil.copy(str(Path(root) / file), FINAL_IMG_DIR / file)

            # ¿Es anotación XML?
            elif ext.lower() == ".xml":
                # El XML se copia sin eliminar duplicados porque cada imagen debe tener un solo xml
                # Si hay duplicados, conservamos el primero
                xml_target = FINAL_ANN_DIR / file
                if not xml_target.exists():
                    shutil.copy(str(Path(root) / file), xml_target)

print(f"Total únicos: {len(seen_files)} imágenes")
print(f"Carpeta final de imágenes: {FINAL_IMG_DIR}")
print(f"Carpeta final de anotaciones: {FINAL_ANN_DIR}")

Total únicos: 8494 imágenes
Carpeta final de imágenes: /Users/gome33773/Documents/MIAA/Proyecto III - Weapon Detection/weapon_dataset/images
Carpeta final de anotaciones: /Users/gome33773/Documents/MIAA/Proyecto III - Weapon Detection/weapon_dataset/annotations


In [5]:
# Extensiones válidas de imágenes
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff",
           ".JPG", ".JPEG", ".PNG", ".BMP", ".GIF", ".TIFF"}

# Contar imágenes
num_imgs = sum(1 for f in FINAL_IMG_DIR.iterdir() if f.suffix in IMG_EXT)

# Contar anotaciones .xml
num_xml = sum(1 for f in FINAL_ANN_DIR.iterdir() if f.suffix.lower() == ".xml")

print(f"Total de imágenes: {num_imgs}")
print(f"Total de anotaciones (XML): {num_xml}")

Total de imágenes: 8494
Total de anotaciones (XML): 8494


**Parseo de anotaciones VOC (XML) PASCAL**

El parseo de anotaciones VOC (XML) se utiliza para leer y extraer la información de las cajas delimitadoras (bounding boxes) y las clases de los objetos anotados en formato PASCAL VOC.
Este proceso convierte esos archivos XML en un formato más usable (como CSV o TXT YOLO) para entrenar modelos de detección de objetos.

In [6]:
def parse_voc_folder(ann_dir, img_dir):
    rows = []
    missing_count = 0

    # Lista de imágenes reales en disco
    image_files = {f.stem: f.name for f in img_dir.iterdir() if f.is_file()}

    for xml_path in sorted(ann_dir.glob('*.xml')):
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Nombre SIN extensión del xml
        xml_filename_raw = root.find('filename').text.strip()
        xml_stem = Path(xml_filename_raw).stem  # solo el nombre sin extensión

        size = root.find('size')
        W = int(size.find('width').text)
        H = int(size.find('height').text)

        # Buscar esa imagen por STEN (sin extensión)
        if xml_stem not in image_files:
            missing_count += 1
            print(f"⚠ No se encontró imagen para: {xml_stem}")
            continue

        # Usar el nombre REAL del archivo (con extensión correcta)
        real_filename = image_files[xml_stem]
        img_path = img_dir / real_filename

        for obj in root.findall('object'):
            cls = obj.find('name').text.strip().lower()
            bnd = obj.find('bndbox')
            xmin = int(float(bnd.find('xmin').text))
            ymin = int(float(bnd.find('ymin').text))
            xmax = int(float(bnd.find('xmax').text))
            ymax = int(float(bnd.find('ymax').text))

            rows.append({
                'image_path': str(img_path),
                'filename': real_filename,
                'xml_path': str(xml_path),
                'width': W, 'height': H,
                'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax,
                'class': cls
            })

    print(f"[INFO] Imágenes no encontradas: {missing_count}")
    return pd.DataFrame(rows)

**Configuración de directorios y parámetros**

In [7]:
# Usar las variables de entorno para configurar las rutas
DATASET_DIR = FINAL_IMG_DIR.parent  # Path al directorio weapon_dataset
ANN_DIR = FINAL_ANN_DIR
IMG_DIR = FINAL_IMG_DIR

# Parseo directo del único dataset final
df = parse_voc_folder(ANN_DIR, IMG_DIR)

clases_validas = ["knife", "pistol"]
df = df[df["class"].isin(clases_validas)]

print(f"[OK] Total de registros cargados: {len(df)}")
df.head(5)

[INFO] Imágenes no encontradas: 0
[OK] Total de registros cargados: 7436


,image_path,filename,xml_path,width,height,xmin,ymin,xmax,ymax,class
0,/Users/gome33773/Documents/MIAA/Proyecto III -...,ABbframe00154.jpg,/Users/gome33773/Documents/MIAA/Proyecto III -...,1920,1090,899,452,1010,568,knife
1,/Users/gome33773/Documents/MIAA/Proyecto III -...,ABbframe00160.jpg,/Users/gome33773/Documents/MIAA/Proyecto III -...,1920,1090,790,425,883,554,knife
2,/Users/gome33773/Documents/MIAA/Proyecto III -...,ABbframe00166.jpg,/Users/gome33773/Documents/MIAA/Proyecto III -...,1920,1090,728,393,830,533,knife
3,/Users/gome33773/Documents/MIAA/Proyecto III -...,ABbframe00169.jpg,/Users/gome33773/Documents/MIAA/Proyecto III -...,1920,1090,761,383,855,521,knife
4,/Users/gome33773/Documents/MIAA/Proyecto III -...,ABbframe00190.jpg,/Users/gome33773/Documents/MIAA/Proyecto III -...,1920,1090,1165,473,1283,553,knife


## Validación de imágenes descargadas de Open Images

Se descargaron imágenes de Open Images V7 en la carpeta `open_images_weapons/` con formato Darknet. A continuación se valida la estructura, el formato de anotaciones y los **cambios necesarios** antes de integrarlas al dataset del modelo YOLO.

In [ ]:
# ============================================================
# VALIDACIÓN DE IMÁGENES DESCARGADAS DE OPEN IMAGES
# ============================================================

OI_BASE = Path(f"{BASE_PATH}/open_images_weapons")

# --- 1. Estructura y conteo ---
print("=" * 60)
print("📁 ESTRUCTURA DE OPEN IMAGES DESCARGADAS")
print("=" * 60)

oi_classes = ['knife', 'handgun', 'rifle']
oi_stats = {}

for cls in oi_classes:
    img_dir = OI_BASE / cls / "images"
    lbl_dir = OI_BASE / cls / "darknet"

    n_imgs = len(list(img_dir.glob("*.jpg"))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0

    oi_stats[cls] = {'images': n_imgs, 'labels': n_lbls, 'match': n_imgs == n_lbls}
    status = "✅" if n_imgs == n_lbls else "⚠️"
    print(f"   {status} {cls}: {n_imgs} imágenes, {n_lbls} labels {'(OK)' if n_imgs == n_lbls else '(MISMATCH)'}")

print(f"\n   Total: {sum(s['images'] for s in oi_stats.values())} imágenes")

# --- 2. Formato de anotaciones (Darknet) ---
print("\n" + "=" * 60)
print("📝 FORMATO DE ANOTACIONES (DARKNET)")
print("=" * 60)

# Leer el archivo de nombres de clases
names_file = OI_BASE / "darknet_obj_names.txt"
if names_file.exists():
    oi_class_names = names_file.read_text().strip().split('\n')
    print(f"   Clases en darknet_obj_names.txt: {oi_class_names}")
    print(f"   Mapeo Open Images: ", end="")
    for i, name in enumerate(oi_class_names):
        print(f"{i}={name}", end="  ")
    print()
else:
    print("   ⚠️ No se encontró darknet_obj_names.txt")

# Mostrar ejemplos de anotaciones
print("\n   Ejemplos de labels:")
for cls in oi_classes:
    lbl_dir = OI_BASE / cls / "darknet"
    if lbl_dir.exists():
        sample_lbl = list(lbl_dir.glob("*.txt"))[:1]
        if sample_lbl:
            content = sample_lbl[0].read_text().strip()
            print(f"   [{cls}] {sample_lbl[0].name}: {content}")

# --- 3. Comparación de mapeo de clases ---
print("\n" + "=" * 60)
print("⚠️  DIFERENCIA CRÍTICA: MAPEO DE CLASES")
print("=" * 60)
print("""
   MODELO ACTUAL (weapons.yaml):      OPEN IMAGES (darknet):
   ─────────────────────────────       ──────────────────────
   0 = knife                           0 = knife
   1 = pistol                          1 = handgun
                                        2 = rifle (NUEVA)

   CAMBIOS NECESARIOS:
   ┌──────────────────────────────────────────────────────────┐
   │ 1. Remapear clase 1 (handgun) → 1 (pistol)              │
   │    Los labels de handgun usan ID=1, que coincide con     │
   │    pistol en nuestro modelo. Misma clase, solo cambia    │
   │    el nombre. ✅ No requiere cambio numérico.            │
   │                                                          │
   │ 2. Clase 0 (knife) → 0 (knife)                          │
   │    Ya coincide. ✅ No requiere cambio.                   │
   │                                                          │
   │ 3. Clase 2 (rifle) → DECISIÓN REQUERIDA:                │
   │    Opción A: Agregar rifle como clase 2 en el modelo     │
   │    Opción B: Remapear rifle → 1 (pistol/arma de fuego)  │
   │    Opción C: Excluir rifles del dataset                  │
   └──────────────────────────────────────────────────────────┘
""")

# --- 4. Verificar integridad de imágenes ---
print("=" * 60)
print("🔍 VERIFICANDO INTEGRIDAD DE IMÁGENES")
print("=" * 60)

corrupt_images = []
grayscale_count = 0
total_checked = 0

for cls in oi_classes:
    img_dir = OI_BASE / cls / "images"
    if not img_dir.exists():
        continue
    for img_path in tqdm(list(img_dir.glob("*.jpg")), desc=f"Verificando {cls}"):
        total_checked += 1
        try:
            img = Image.open(img_path)
            img.verify()  # Verificar integridad
            img = Image.open(img_path)  # Reabrir tras verify
            if img.mode == 'L':
                grayscale_count += 1
        except Exception as e:
            corrupt_images.append((str(img_path), str(e)))

print(f"\n   Total verificadas: {total_checked}")
print(f"   Corruptas: {len(corrupt_images)}")
print(f"   Escala de grises: {grayscale_count}")
if corrupt_images:
    print("   Imágenes corruptas:")
    for path, err in corrupt_images[:5]:
        print(f"     ⚠️ {path}: {err}")

# --- 5. Validar formato de labels ---
print("\n" + "=" * 60)
print("🏷️  VALIDANDO FORMATO DE LABELS")
print("=" * 60)

invalid_labels = []
class_distribution = {0: 0, 1: 0, 2: 0}

for cls in oi_classes:
    lbl_dir = OI_BASE / cls / "darknet"
    if not lbl_dir.exists():
        continue
    for lbl_path in lbl_dir.glob("*.txt"):
        content = lbl_path.read_text().strip()
        if not content:
            invalid_labels.append((str(lbl_path), "Archivo vacío"))
            continue
        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) != 5:
                invalid_labels.append((str(lbl_path), f"Formato incorrecto: {line}"))
                continue
            try:
                cls_id = int(parts[0])
                coords = [float(p) for p in parts[1:]]
                class_distribution[cls_id] = class_distribution.get(cls_id, 0) + 1
                # Verificar que coordenadas estén en rango [0, 1]
                if any(c < 0 or c > 1 for c in coords):
                    invalid_labels.append((str(lbl_path), f"Coords fuera de rango: {line}"))
            except ValueError:
                invalid_labels.append((str(lbl_path), f"Error de parseo: {line}"))

print(f"   Labels inválidos: {len(invalid_labels)}")
print(f"   Distribución de clases en anotaciones:")
for cls_id, count in sorted(class_distribution.items()):
    name = oi_class_names[cls_id] if cls_id < len(oi_class_names) else "desconocida"
    print(f"     ID {cls_id} ({name}): {count} anotaciones")
if invalid_labels:
    print(f"\n   Primeros labels con problemas:")
    for path, err in invalid_labels[:5]:
        print(f"     ⚠️ {Path(path).name}: {err}")

📁 ESTRUCTURA DE OPEN IMAGES DESCARGADAS
   ✅ knife: 537 imágenes, 537 labels (OK)
   ✅ handgun: 281 imágenes, 281 labels (OK)
   ✅ rifle: 1000 imágenes, 1000 labels (OK)

   Total: 1818 imágenes

📝 FORMATO DE ANOTACIONES (DARKNET)
   Clases en darknet_obj_names.txt: ['knife', 'handgun', 'rifle']
   Mapeo Open Images: 0=knife  1=handgun  2=rifle  

   Ejemplos de labels:
   [knife] 0320662d45d55cdf.txt: 0 0.6915625 0.37281249999999994 0.44187499999999996 0.45562499999999995
   [handgun] 386f7887405c6796.txt: 1 0.5140625 0.5194610000000001 0.815625 0.583832
   [rifle] 00c04a1be1f153f0.txt: 2 0.5134375 0.5845835 0.971875 0.539167

⚠️  DIFERENCIA CRÍTICA: MAPEO DE CLASES

   MODELO ACTUAL (weapons.yaml):      OPEN IMAGES (darknet):
   ─────────────────────────────       ──────────────────────
   0 = knife                           0 = knife
   1 = pistol                          1 = handgun
                                        2 = rifle (NUEVA)

   CAMBIOS NECESARIOS:
   ┌──────────────

Verificando rifle: 100%|██████████| 1000/1000 [00:01<00:00, 663.38it/s]



   Total verificadas: 1818
   Corruptas: 0
   Escala de grises: 0

🏷️  VALIDANDO FORMATO DE LABELS
   Labels inválidos: 0
   Distribución de clases en anotaciones:
     ID 0 (knife): 729 anotaciones
     ID 1 (handgun): 330 anotaciones
     ID 2 (rifle): 1049 anotaciones


## Resumen de cambios necesarios para la integración

Tras la validación, los cambios requeridos para integrar las imágenes de Open Images al dataset YOLO existente son:

### Cambios obligatorios

| # | Cambio | Detalle |
|---|--------|---------|
| 1 | **Remapeo de clase `handgun` (ID 1 → 1)** | Coincide numéricamente con `pistol` en nuestro modelo. No requiere cambio en los archivos `.txt`, solo consistencia semántica. |
| 2 | **Clase `knife` (ID 0 → 0)** | Ya coincide. Sin cambios necesarios. |
| 3 | **Decisión sobre `rifle` (ID 2)** | **Opción A**: agregarlo como nueva clase 2 al `weapons.yaml`. **Opción B**: reasignar ID 2 → 1 para tratarlo como arma de fuego genérica. **Opción C**: excluir rifles. |
| 4 | **Renombrar carpeta `darknet/` → copiar a `labels/`** | Los labels en formato Darknet ya son formato YOLO (`class xc yc w h`). Solo hay que moverlos a la estructura `yolo_dataset/labels/train/` y `val/`. |
| 5 | **Mover imágenes** | Copiar de `open_images_weapons/{clase}/images/` a `yolo_dataset/images/train/` y `val/` (con split 80/20). |

### Opcional pero recomendado

| # | Acción | Razón |
|---|--------|-------|
| 6 | Verificar y eliminar imágenes corruptas | Evitar errores durante el entrenamiento |
| 7 | Convertir imágenes en escala de grises a RGB | YOLO espera 3 canales |
| 8 | Actualizar `weapons.yaml` | Si se incluye rifle, agregar `2: rifle` al mapeo de clases |

## Integración de Open Images al dataset YOLO

Se incluirá **rifle como clase nueva (ID 2)**. El mapeo final del modelo será:

| ID | Clase | Fuente |
|----|-------|--------|
| 0 | knife | DaSCI + Open Images |
| 1 | pistol | DaSCI (pistol) + Open Images (handgun) |
| 2 | rifle | Open Images (nuevo) |

Los IDs de Open Images Darknet (`0=knife, 1=handgun, 2=rifle`) ya coinciden numéricamente con el mapeo objetivo, por lo que **no se requiere remapeo de labels**. Solo se necesita:
1. Limpiar imágenes corruptas y convertir escala de grises → RGB
2. Copiar imágenes y labels a `yolo_dataset/` con split 80/20
3. Actualizar `weapons.yaml` con la nueva clase

In [ ]:
# ============================================================
# INTEGRACIÓN DE OPEN IMAGES AL DATASET YOLO
# Rifle como clase nueva (ID 2)
# ============================================================

import random

OI_BASE = Path(f"{BASE_PATH}/open_images_weapons")
oi_classes_dirs = ['knife', 'handgun', 'rifle']

# --- 1. Limpiar imágenes corruptas y convertir escala de grises ---
print("=" * 60)
print("🧹 PASO 1: Limpieza de imágenes")
print("=" * 60)

removed_count = 0
converted_count = 0

for cls in oi_classes_dirs:
    img_dir = OI_BASE / cls / "images"
    lbl_dir = OI_BASE / cls / "darknet"
    if not img_dir.exists():
        continue

    for img_path in list(img_dir.glob("*.jpg")):
        try:
            img = Image.open(img_path)
            img.verify()
            img = Image.open(img_path)

            # Convertir escala de grises a RGB
            if img.mode != 'RGB':
                img = img.convert('RGB')
                img.save(img_path)
                converted_count += 1

        except Exception:
            # Eliminar imagen corrupta y su label
            img_path.unlink(missing_ok=True)
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            lbl_path.unlink(missing_ok=True)
            removed_count += 1

print(f"   Imágenes corruptas eliminadas: {removed_count}")
print(f"   Imágenes convertidas a RGB: {converted_count}")

# --- 2. Copiar imágenes y labels al dataset YOLO (split 80/20) ---
print("\n" + "=" * 60)
print("📂 PASO 2: Integración al dataset YOLO (split 80/20)")
print("=" * 60)

# Usar las mismas rutas del dataset YOLO existente
TRAIN_IMG_DIR = YOLO_DIR / "images" / "train"
VAL_IMG_DIR = YOLO_DIR / "images" / "val"
TRAIN_LAB_DIR = YOLO_DIR / "labels" / "train"
VAL_LAB_DIR = YOLO_DIR / "labels" / "val"

total_copied = {'train': 0, 'val': 0}
skipped = 0

for cls in oi_classes_dirs:
    img_dir = OI_BASE / cls / "images"
    lbl_dir = OI_BASE / cls / "darknet"
    if not img_dir.exists() or not lbl_dir.exists():
        continue

    # Obtener pares imagen-label válidos
    pairs = []
    for img_path in img_dir.glob("*.jpg"):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            pairs.append((img_path, lbl_path))

    # Shuffle y split
    random.seed(42)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * 0.8)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    for split_name, split_pairs, img_dest, lbl_dest in [
        ('train', train_pairs, TRAIN_IMG_DIR, TRAIN_LAB_DIR),
        ('val', val_pairs, VAL_IMG_DIR, VAL_LAB_DIR)
    ]:
        for img_path, lbl_path in split_pairs:
            # Prefijo para evitar colisiones de nombres
            new_name = f"oi_{cls}_{img_path.stem}"
            dst_img = img_dest / f"{new_name}.jpg"
            dst_lbl = lbl_dest / f"{new_name}.txt"

            # Saltar si ya existe
            if dst_img.exists():
                skipped += 1
                continue

            shutil.copy2(str(img_path), str(dst_img))
            shutil.copy2(str(lbl_path), str(dst_lbl))
            total_copied[split_name] += 1

    print(f"   {cls}: {len(train_pairs)} train + {len(val_pairs)} val")

print(f"\n   Total copiado → train: {total_copied['train']}, val: {total_copied['val']}")
if skipped > 0:
    print(f"   Salteados (ya existían): {skipped}")

# --- 3. Actualizar weapons.yaml ---
print("\n" + "=" * 60)
print("📝 PASO 3: Actualizando weapons.yaml")
print("=" * 60)

yaml_text = f"""
path: {YOLO_DIR}

train: images/train
val: images/val

names:
  0: knife
  1: pistol
  2: rifle
"""

with open("weapons.yaml", "w") as f:
    f.write(yaml_text)

print("   weapons.yaml actualizado:")
print("   0: knife")
print("   1: pistol")
print("   2: rifle  ← NUEVA CLASE")

# --- 4. Verificación final ---
print("\n" + "=" * 60)
print("✅ PASO 4: Verificación final del dataset integrado")
print("=" * 60)

for split_name, img_d, lbl_d in [("train", TRAIN_IMG_DIR, TRAIN_LAB_DIR), ("val", VAL_IMG_DIR, VAL_LAB_DIR)]:
    n_imgs = len([f for f in img_d.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
    n_lbls = len([f for f in lbl_d.iterdir() if f.suffix == '.txt'])
    status = "✅" if n_imgs == n_lbls else "⚠️"
    print(f"   {status} {split_name}: {n_imgs} imágenes, {n_lbls} labels")

# Contar distribución de clases en todo el dataset
print("\n   Distribución de clases (dataset completo):")
class_counts = {0: 0, 1: 0, 2: 0}
class_names_final = {0: 'knife', 1: 'pistol', 2: 'rifle'}

for lbl_dir in [TRAIN_LAB_DIR, VAL_LAB_DIR]:
    for lbl_path in lbl_dir.glob("*.txt"):
        for line in lbl_path.read_text().strip().split('\n'):
            if line.strip():
                cls_id = int(line.strip().split()[0])
                class_counts[cls_id] = class_counts.get(cls_id, 0) + 1

for cls_id in sorted(class_counts):
    name = class_names_final.get(cls_id, 'desconocida')
    count = class_counts[cls_id]
    print(f"     {cls_id} ({name}): {count} anotaciones")

total_anns = sum(class_counts.values())
total_imgs = sum(len([f for f in d.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']])
                 for d in [TRAIN_IMG_DIR, VAL_IMG_DIR])
print(f"\n   📊 TOTAL: {total_imgs} imágenes, {total_anns} anotaciones, 3 clases")
print("=" * 60)

🧹 PASO 1: Limpieza de imágenes
   Imágenes corruptas eliminadas: 0
   Imágenes convertidas a RGB: 0

📂 PASO 2: Integración al dataset YOLO (split 80/20)
   knife: 429 train + 108 val
   handgun: 224 train + 57 val
   rifle: 800 train + 200 val

   Total copiado → train: 0, val: 0
   Salteados (ya existían): 1818

📝 PASO 3: Actualizando weapons.yaml
   weapons.yaml actualizado:
   0: knife
   1: pistol
   2: rifle  ← NUEVA CLASE

✅ PASO 4: Verificación final del dataset integrado
   ✅ train: 21940 imágenes, 21940 labels
   ✅ val: 4393 imágenes, 4393 labels

   Distribución de clases (dataset completo):
     0 (knife): 18865 anotaciones
     1 (pistol): 8384 anotaciones
     2 (rifle): 1049 anotaciones

   📊 TOTAL: 26333 imágenes, 28298 anotaciones, 3 clases
